# Interpretable Models Comparison

Endgame provides 30+ glass-box models that are fully transparent and auditable,
all sharing the same scikit-learn-compatible API (`fit`, `predict`, `predict_proba`).

In this notebook we train four interpretable classifiers on a breast cancer dataset
and compare their accuracy:

1. **EBM** (Explainable Boosting Machine) -- a modern GAM with automatic interactions
2. **RuleFit** -- tree-derived rules combined via Lasso regression
3. **MARS** -- piecewise-linear splines that discover knots automatically
4. **C5.0** -- the successor to C4.5, a classic interpretable decision tree

Each model produces human-readable explanations while achieving accuracy
competitive with black-box gradient boosting.

In [ ]:
import endgame as eg
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## Load Data

We use the Wisconsin Breast Cancer dataset (569 samples, 30 numeric features,
binary target). This is a well-studied benchmark where interpretable models
are especially valuable -- clinicians need to understand *why* a prediction
was made.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {np.unique(y_train)} (0=malignant, 1=benign)")

In [ ]:
# Store results for the final comparison
results = {}

## 1. Explainable Boosting Machine (EBM)

EBMs are a modern form of Generalized Additive Model (GAM) that use
cyclic gradient boosting over each feature independently, then adds
pairwise interaction terms. They achieve accuracy close to full
gradient boosting while remaining fully interpretable.

Setting `interactions=10` allows up to 10 automatically detected
pairwise interaction terms.

In [ ]:
from endgame.models import EBMClassifier

ebm = EBMClassifier(interactions=10)
ebm.fit(X_train, y_train)

ebm_acc = accuracy_score(y_test, ebm.predict(X_test))
results["EBM"] = ebm_acc
print(f"EBM Test Accuracy: {ebm_acc:.4f}")

In [ ]:
# EBM feature importances -- shows how much each feature
# contributes to predictions on average
importances = ebm.feature_importances_
top_idx = np.argsort(importances)[-10:][::-1]

print("Top 10 EBM features by importance:")
for i in top_idx:
    print(f"  {X_train.columns[i]:>30s}: {importances[i]:.4f}")

## 2. RuleFit

RuleFit (Friedman & Popescu, 2008) first fits a tree ensemble, extracts
human-readable rules from every tree, then selects the most important rules
via L1-regularized (Lasso) regression. The result is a sparse set of
if-then rules that explain the prediction.

In [ ]:
from endgame.models.rules.rulefit import RuleFitClassifier

rulefit = RuleFitClassifier(random_state=42)
rulefit.fit(X_train.values, y_train)

rulefit_acc = accuracy_score(y_test, rulefit.predict(X_test.values))
results["RuleFit"] = rulefit_acc
print(f"RuleFit Test Accuracy: {rulefit_acc:.4f}")

In [ ]:
# Show the top rules discovered by RuleFit
rules = rulefit.get_rules(exclude_zero_coef=True, sort_by="importance")
print(f"Total active rules: {len(rules)}\n")
print("Top 5 rules by importance:")
for i, rule in enumerate(rules[:5], 1):
    print(f"  Rule {i}: {rule['rule']}")
    print(f"    coefficient={rule['coefficient']:.4f}, "
          f"support={rule['support']:.3f}, "
          f"importance={rule['importance']:.4f}")
    print()

## 3. MARS (Multivariate Adaptive Regression Splines)

MARS (Friedman, 1991) builds a piecewise-linear model by automatically
discovering *knots* -- thresholds where the relationship between a feature
and the target changes slope. The classifier version applies logistic
regression on the discovered basis functions.

In [ ]:
from endgame.models.linear.mars import MARSClassifier

mars = MARSClassifier(max_degree=1)
mars.fit(X_train.values, y_train)

mars_acc = accuracy_score(y_test, mars.predict(X_test.values))
results["MARS"] = mars_acc
print(f"MARS Test Accuracy: {mars_acc:.4f}")

In [ ]:
# MARS summary shows the basis functions (hinges) and their coefficients
print(mars.summary())

## 4. C5.0 Decision Tree

C5.0 (Quinlan, 1993) is the commercial successor to C4.5 and one of the
most widely deployed decision tree algorithms. It uses gain ratio for
splitting, handles missing values natively, and applies both local and
global pruning for compact, interpretable trees.

We use `use_rust=False` to use the pure-Python implementation, which
does not require the optional `c50-rs` Rust backend.

In [ ]:
from endgame.models.trees.c50 import C50Classifier

c50 = C50Classifier(use_rust=False, random_state=42)
c50.fit(X_train.values, y_train)

c50_acc = accuracy_score(y_test, c50.predict(X_test.values))
results["C5.0"] = c50_acc
print(f"C5.0 Test Accuracy: {c50_acc:.4f}")

In [ ]:
# C5.0 feature importances based on split gains
importances_c50 = c50.feature_importances_
top_idx_c50 = np.argsort(importances_c50)[-10:][::-1]

print("Top 10 C5.0 features by split gain:")
for i in top_idx_c50:
    print(f"  {X_train.columns[i]:>30s}: {importances_c50[i]:.4f}")

## Model Comparison

Let's compare all four interpretable models side by side.

In [ ]:
comparison = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": [f"{v:.4f}" for v in results.values()],
    "Interpretability": [
        "Feature shape functions + interactions",
        "Sparse set of if-then rules",
        "Piecewise-linear hinge functions",
        "Pruned decision tree",
    ],
})

print("=" * 70)
print("Interpretable Models -- Breast Cancer Dataset")
print("=" * 70)
print(comparison.to_string(index=False))
print("=" * 70)

## Summary

**Key takeaway:** Interpretable models achieve near-GBDT accuracy while
remaining fully auditable. Each model family offers a different *form*
of explanation:

| Model   | Explanation Form                          |
|---------|-------------------------------------------|
| EBM     | Per-feature shape plots + interaction heatmaps |
| RuleFit | Weighted if-then rules                    |
| MARS    | Piecewise-linear basis functions with knots |
| C5.0    | Compact decision tree with gain-ratio splits |

Choose the model whose explanation format best matches your audience:
- **EBM** for data scientists who want rich global/local explanations
- **RuleFit** when stakeholders need simple business rules
- **MARS** for continuous, smooth feature effects
- **C5.0** for a single, printable decision tree

Next steps:
- Try `03_automl.ipynb` for automated model selection and ensembling
- Use `eg.explain()` for post-hoc explanations of any black-box model